In [3]:
import torch
import torch.nn as nn
from spin_lattices import KagomeLattice, SquareLattice1Diag, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from loguru import logger
from slater_determinant import SlaterDeterminant, tight_binding_init, HalfSpaceProjector, TwoHalvesOuterProduct
from pathlib import Path
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import torch.nn.utils.parametrize as parametrize
from torch.nn.utils.parametrizations import orthogonal

def sign_overlap(ground_state, predict_signs):
    probs = ground_state**2
    return torch.dot(ground_state, predict_signs * torch.abs(ground_state)) / probs.sum()

In [4]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=1, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)

ground_state_np = np.real_if_close(system.get_ground_state_in_canonical_basis())
ground_state = torch.from_numpy(ground_state_np)

2023-04-26 15:35:44.436 | DEBUG    | heisenberg_hamiltonians:__init__:427 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-04-26 15:35:44.437 | DEBUG    | heisenberg_hamiltonians:__init__:448 - number_spins=24
2023-04-26 15:35:44.443 | DEBUG    | heisenberg_hamiltonians:__init__:458 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-26 15:35:44.532 | DEBUG    | heisenberg_hamiltonians:__init__:467 - Hilbert space dimension is 85662
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-26 15:35:44.614 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-True-1-1.pickle
2023-04-26 15:35:44.617 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -42.8245991763
2023-04-26 15:35:44.618 | DEBUG    | heisenberg

In [5]:
# with torch.no_grad():
#     det.f.copy_(
#         nn.Parameter(
#             torch.randn(system.number_spins, system.number_spins, dtype=torch.float64)
#             / np.sqrt(system.number_spins)
#         )
#     )

eps_train = 0.001
test_size = 10000
epochs = 1000
batch_size = 64
lr = 1e-3
scaling = 10000000


for run in range(10):
    dataset_seed = run

    np.random.seed(dataset_seed)
    train_set_numpy = np.random.choice(
        len(system.canonical_basis.states),
        int(eps_train * len(system.canonical_basis.states)),
        replace=False,
        p=ground_state**2,
    )

    train_set = torch.from_numpy(train_set_numpy)
    logger.debug(f"{len(train_set)=}")

    target = (ground_state[train_set] > 0).double()

    rest_set_np = np.setdiff1d(np.arange(len(system.canonical_basis.states)), train_set_numpy)
    rest_probs = ground_state_np[rest_set_np] ** 2
    rest_probs /= rest_probs.sum()
    test_set = torch.from_numpy(np.random.choice(rest_set_np, test_size, replace=False))


    logger.debug(f"{run=}")
    writer = SummaryWriter(
        log_dir=(
            f"experiments/2022_04_24/{datetime.now().strftime('%H_%M_%S')}"
            f"_{eps_train=}_{batch_size=}_{lr=}"
        #    f"_{initialization=}"
        #    f"_{scaling=}_{keep_symmetries=}"
        )
    )

    torch.manual_seed(run)
    det = orthogonal(SlaterDeterminant(
        system.lattice,
        system.canonical_basis,
        initialization='randn',
        sign_cache_dir=Path("signs_cache"),
    ), name="f")

    parametrize.register_parametrization(det, "f", TwoHalvesOuterProduct())

    n_batches = len(train_set) // batch_size

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(det.parameters(), lr=lr)

    epoch = 0
    logger.debug(f"{n_batches=}")
    for epoch in range(epochs):  # loop over the dataset multiple times
        i = None
        loss = None


        for i in range(n_batches):
            x = train_set[i * batch_size : (i + 1) * batch_size]
            y = target[i * batch_size : (i + 1) * batch_size]

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            det_output = det(x) * scaling
            outputs = torch.sigmoid(det_output)
        
            # print(f"{det_output=}")
            # print(f"{outputs=}")
            # 1/0
            

            #        print(det(x))
            loss = criterion(outputs, y)
            loss.backward()
            #        print(det.f.grad.norm().item())

            optimizer.step()

        assert loss is not None

        overlap_train = sign_overlap(ground_state[train_set], torch.sign(det(train_set) * scaling))
        overlap_test = sign_overlap(ground_state[test_set], torch.sign(det(test_set) * scaling))
        overlap_train_unweighted = torch.sign(ground_state[train_set] * det(train_set)).mean()

        writer.add_scalar("Loss/train", loss.item(), epoch)
        writer.add_scalar("Overlap/train", overlap_train.item(), epoch)
        writer.add_scalar("Overlap/test", overlap_test.item(), epoch)
        writer.add_scalar("Overlap/train/unweighted", overlap_train_unweighted.item(), epoch)

        # writer.add_scalar("Det_output/std/train", det_output.std().item(), epoch)

        # log.append(
        #     {
        #         "epoch": epoch,
        #         "loss": loss.item(),
        #         "overlap_train": overlap_train.item(),
        #         "overlap_test": overlap_test.item(),
        #     }
        # )
        # logger.debug(
        #     f"Epoch {epoch} loss: {loss.item():.4f} overlap_train: {overlap_train.item():.4f} "
        #     f"overlap_test: {overlap_test.item():.4f}"
        # )
            

2023-04-26 15:35:45.418 | DEBUG    | __main__:<module>:29 - len(train_set)=2704
2023-04-26 15:35:45.848 | DEBUG    | __main__:<module>:39 - run=0
2023-04-26 15:35:48.147 | DEBUG    | slater_determinant:__init__:160 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-26 15:35:48.171 | DEBUG    | __main__:<module>:66 - n_batches=42
2023-04-26 15:38:30.699 | DEBUG    | __main__:<module>:29 - len(train_set)=2704
2023-04-26 15:38:30.910 | DEBUG    | __main__:<module>:39 - run=1
2023-04-26 15:38:33.192 | DEBUG    | slater_determinant:__init__:160 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-26 15:38:33.205 | DEBUG    | __main__:<module>:66 - n_batches=42


KeyboardInterrupt: 

In [5]:
U, S, V = torch.svd(det.f)

In [6]:
S

tensor([1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        3.3150e-16, 2.8560e-16, 2.4958e-16, 2.0339e-16, 1.6968e-16, 1.5805e-16,
        1.4284e-16, 9.9362e-17, 6.1774e-17, 4.6013e-17, 3.7269e-17, 2.5638e-18],
       dtype=torch.float64, grad_fn=<LinalgSvdBackward0>)